# T3 — Counts versus anchors

**Facts used** (classical; catalog c05, c08, c11, c14, c15). The organizing
sentence: *the integer is structural, the anchor is medium* (X-10's
"order + number"; X-11's "number enters only as count").

1. Koenig (Goldstein ch. 1): $L = R_{cm}\times P + L_{rel}$ and
   $T = \tfrac12 M V_{cm}^2 + T_{rel}$ for any particle system (c08, c14); a
   uniform field exerts no torque about the CM (c15). The split is an identity
   in the masses - no medium enters.
2. Toomre (1964; Binney & Tremaine eq. 6.55): the fluid-disk WKB dispersion
   $\omega^2 = \kappa^2 - 2\pi G\Sigma k + k^2\sigma^2$ has growing modes iff
   $\sigma\kappa/(\pi G\Sigma) < 1$ (c05). The threshold **1** is the
   discriminant of a quadratic; $G, \Sigma, \kappa, \sigma$ are the anchors.
3. Schrödinger (1914): the nearest-neighbour chain has
   $\omega(k) = 2\sqrt{J/m}\,|\sin(k/2)|$ (c11). A ring of $N$ masses has
   exactly $N$ modes whatever $J/m$; $\sqrt{J/m}$ only scales them.

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

## 1. Koenig: an identity in the masses

In [ ]:
def koenig(seed):
    random.seed(seed)
    n = 7
    m = [random.uniform(0.5, 2) for _ in range(n)]
    r = [[random.uniform(-1, 1) for _ in range(3)] for _ in range(n)]
    v = [[random.uniform(-1, 1) for _ in range(3)] for _ in range(n)]
    M = sum(m)
    R = [sum(m[i] * r[i][k] for i in range(n)) / M for k in range(3)]
    V = [sum(m[i] * v[i][k] for i in range(n)) / M for k in range(3)]
    cross = lambda a, b: [a[1] * b[2] - a[2] * b[1], a[2] * b[0] - a[0] * b[2], a[0] * b[1] - a[1] * b[0]]
    L = [sum(m[i] * cross(r[i], v[i])[k] for i in range(n)) for k in range(3)]
    L_rel = [sum(m[i] * cross([r[i][j] - R[j] for j in range(3)], [v[i][j] - V[j] for j in range(3)])[k] for i in range(n)) for k in range(3)]
    L_cm = cross(R, [M * V[k] for k in range(3)])
    T = sum(0.5 * m[i] * sum(v[i][k] ** 2 for k in range(3)) for i in range(n))
    T_rel = sum(0.5 * m[i] * sum((v[i][k] - V[k]) ** 2 for k in range(3)) for i in range(n))
    T_cm = 0.5 * M * sum(V[k] ** 2 for k in range(3))
    return L, L_rel, L_cm, T, T_rel, T_cm

for seed in (1, 2, 3):
    L, L_rel, L_cm, T, T_rel, T_cm = koenig(seed)
    errL = max(abs(L[k] - L_rel[k] - L_cm[k]) for k in range(3))
    print(f"seed {seed}: |L - (R x P + L_rel)| = {errL:.1e}   |T - (M V^2/2 + T_rel)| = {abs(T - T_rel - T_cm):.1e}")

In [ ]:
def check(keep_cm_term=True):
    L, L_rel, L_cm, T, T_rel, T_cm = koenig(5)
    w = 1.0 if keep_cm_term else 0.0
    return max(abs(L[k] - L_rel[k] - w * L_cm[k]) for k in range(3)) < 1e-12 and abs(T - T_rel - w * T_cm) < 1e-12

falsify(check, {"omit-cm-term": lambda: {"keep_cm_term": False}})

## 2. Toomre: the 1 is a discriminant

In [ ]:
def min_omega2(sigma, G=1.0, Sigma=1.0, kappa=1.0):
    # omega^2(k) = kappa^2 - 2 pi G Sigma k + k^2 sigma^2; a quadratic in k with minimum at k* = pi G Sigma / sigma^2
    ks = [0.001 * j for j in range(1, 20000)]
    numeric = min(kappa ** 2 - 2 * math.pi * G * Sigma * k + k * k * sigma * sigma for k in ks)
    closed = kappa ** 2 - (math.pi * G * Sigma / sigma) ** 2
    return numeric, closed

for anchors in ((1.0, 1.0, 1.0), (2.0, 0.5, 3.0), (0.3, 4.0, 0.7)):
    G, Sigma, kappa = anchors
    print(f"anchors G={G}, Sigma={Sigma}, kappa={kappa}:")
    for Q in (0.9, 0.99, 1.01, 1.1):
        sigma = Q * math.pi * G * Sigma / kappa
        num, clo = min_omega2(sigma, G, Sigma, kappa)
        print(f"   Q = {Q:<5} min omega^2 = {num:+.4f} (closed form {clo:+.4f}) -> {'unstable' if num < 0 else 'stable'}")

In [ ]:
def check(threshold=1.0):
    ok = True
    for G, Sigma, kappa in ((1.0, 1.0, 1.0), (2.0, 0.5, 3.0), (0.3, 4.0, 0.7)):
        for Q in (0.5, 0.9, 0.99, 1.01, 1.1, 1.5):
            sigma = Q * math.pi * G * Sigma / kappa
            ok &= (min_omega2(sigma, G, Sigma, kappa)[0] < 0) == (Q < threshold)
    return ok

falsify(check, {"threshold-2": lambda: {"threshold": 2.0}})

## 3. The chain: $N$ modes, one anchor

In [ ]:
def chain_modes(N, J, m):
    # eigenvalues of the ring Laplacian on plane waves k = 2 pi n / N
    return sorted(math.sqrt((2 - 2 * math.cos(2 * math.pi * n / N)) * J / m) for n in range(N))

N = 12
for J, m in ((1.0, 1.0), (4.0, 1.0), (1.0, 9.0)):
    w = chain_modes(N, J, m)
    scaled = [x / math.sqrt(J / m) for x in w]
    print(f"J/m = {J / m:<5}: {len(w)} modes; omega/sqrt(J/m) = {[round(s, 4) for s in scaled[:6]]} ...")
ks = [2 * math.pi * n / N for n in range(N // 2 + 1)]
print(termplot.plot_xy([(k, 2 * abs(math.sin(k / 2))) for k in ks] + [(k, k) for k in ks], width=50, height=12,
                       title="omega/sqrt(J/m): chain (curve) vs continuum line", xlabel="k", ylabel="omega"))

In [ ]:
def check(law="chain"):
    worst = 0.0
    for n in (1, 5, 13, 31):
        k = 2 * math.pi * n / 64
        num = math.sqrt(2 - 2 * math.cos(k))
        claim = k if law == "continuum" else 2 * abs(math.sin(k / 2))
        worst = max(worst, abs(num - claim) / num)
    return worst < 1e-12

falsify(check, {"continuum-law": lambda: {"law": "continuum"}})

## Falsifier: every catalog mutant used here must fail

In [ ]:
for entry in ("c05_toomre_fluid", "c08_koenig_angular_momentum", "c14_koenig_kinetic_energy",
              "c15_cm_torque_uniform_gravity", "c11_chain_dispersion"):
    rc, _ = catalog(entry)
    assert rc == 0, entry
    rc, out = catalog(entry, mutant=True)
    mutant_must_fail(entry, rc, out)